# Notebook 03 — Alignment

**Project:** Iqraa AI — Quranic recitation correction for Riwayat Qaloon an Nafi  
**Runtime required:** T4 GPU (or better)

## The Core Problem

Notebook 01 produced 114 long WAV files — one per surah. Surah 2 (Al-Baqarah)
is nearly two hours of audio. The model cannot train on a two-hour clip.
It needs short, labelled segments — one per ayah, typically 2–15 seconds.

This notebook cuts each surah WAV into individual ayah-level clips.

## How: Forced Alignment with WhisperX

Forced alignment is the process of finding exactly where in an audio file
each word is spoken. WhisperX is a two-stage tool: first, OpenAI Whisper
produces a rough transcript with approximate timestamps; second, a Wav2Vec2
model trained for alignment refines those timestamps to the exact millisecond.
We ignore Whisper's transcription completely — we only use its timing output.
We replace all text with our verified Qaloon labels from
`qalon_canonical.json`. This is important: if we used Whisper's transcription,
it would give us Hafs text, not Qaloon text, and the training labels would
be wrong for roughly 10% of the corpus.

## Checkpointing

Processing 114 surahs takes 2–4 hours. A checkpoint file is saved after every
surah. If Colab disconnects, re-running this notebook skips completed surahs
and resumes from where it stopped — no work is lost or duplicated.

## Cell 2 — Mount Google Drive and Configure All Paths

`CHECKPOINT_DIR` is new in this notebook. One JSON file per surah is written
here immediately after that surah finishes. On restart, completed surahs are
detected by checking whether their checkpoint file exists.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_BASE     = '/content/drive/MyDrive/IqraaAI_Dataset'
WAV_DIR        = DRIVE_BASE + '/processed/wav'
SEGMENTS_DIR   = DRIVE_BASE + '/processed/segments'
CHECKPOINT_DIR = DRIVE_BASE + '/processed/alignment_checkpoints'
TEXT_FILE      = DRIVE_BASE + '/text/qalon_canonical.json'
MANIFEST_FILE  = DRIVE_BASE + '/processed/pairing_manifest.json'

os.makedirs(SEGMENTS_DIR,   exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print('Drive mounted.')
print(f'WAV dir        : {WAV_DIR}')
print(f'Segments dir   : {SEGMENTS_DIR}')
print(f'Checkpoint dir : {CHECKPOINT_DIR}')

## Cell 3 — What WhisperX Is and Why We Use It

WhisperX is a two-stage tool. First, OpenAI Whisper produces a rough
transcript with approximate timestamps. Second, a Wav2Vec2 model refines
those timestamps to the exact millisecond. We ignore Whisper's transcription
completely — we only use its timing output. We replace all text with our
verified Qaloon labels from `qalon_canonical.json`. This is important: if we
used Whisper's transcription, it would give us Hafs text, not Qaloon text.

### Why WhisperX over other aligners?

- `large-v3` model has strong Arabic support including diacritised text
- The Wav2Vec2 alignment stage gives millisecond-precision word boundaries
- Works out-of-the-box on Colab without installing system-level tools
- The forced-alignment step does not require a reference transcript — it
  discovers boundaries from acoustics alone

### What we get

For each surah, WhisperX produces a list like:

```
[{"word": "بسم", "start": 0.42, "end": 0.91},
 {"word": "الله", "start": 0.94, "end": 1.48}, ...]
```

We then group those word timestamps into ayah-sized buckets using the word
count from `qalon_canonical.json`, and cut the audio at the boundaries.

## Cell 4 — Install Dependencies and Verify GPU

WhisperX requires a GPU for the alignment step. We verify a GPU is available
before proceeding. If the check fails, change the runtime type to T4 GPU
via Runtime → Change runtime type.

In [ ]:
!pip install -q whisperx
!pip install -q torch torchaudio

import torch
print(f'GPU available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device        : {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('No GPU detected. Change runtime to T4 GPU and re-run.')

## Cell 5 — The Checkpointing System

Processing 114 surahs takes 2–4 hours. Colab may disconnect during this time.
To handle this safely, we save a checkpoint file after completing each surah.
If the notebook restarts, it reads the checkpoint directory and skips surahs
already done. You can safely re-run this notebook multiple times — it will
never duplicate work or overwrite completed segments.

### Checkpoint format

Each checkpoint is a JSON file at `CHECKPOINT_DIR/001.json`, `002.json`, etc.
It contains the list of segment dicts produced for that surah:

```json
[
  {
    "ayah_key": "001001",
    "surah": 1,
    "ayah": 1,
    "text": "الْحَمْدُ لِلهِ رَبِّ الْعَالَمِينَ",
    "wav_path": ".../segments/001_001.wav",
    "duration_sec": 3.24,
    "alignment_method": "whisperx"
  },
  ...
]
```

The `alignment_method` field records whether the segment boundary came from
WhisperX word timestamps (`whisperx`) or the equal-time fallback
(`equal_time`). This lets us audit which surahs were hard to align.

## Cell 6 — Load Text, Check Checkpoint State

We load `qalon_canonical.json`, scan the checkpoint directory for completed
surahs, and report how many are done vs remaining. Running this cell on a
fresh session immediately shows the resume state without touching any models.

In [ ]:
import json
import unicodedata

# Inline normalisation — identical to src/normalize_text.py
_REMOVE_CHARS = frozenset(
    [chr(c) for c in range(0x06D6, 0x06DD)]
    + [chr(c) for c in range(0x06DF, 0x06E5)]
    + [chr(0x06E7), chr(0x06E8)]
    + [chr(c) for c in range(0x06EA, 0x06EE)]
    + [chr(c) for c in range(0x0660, 0x066A)]
    + [chr(0x0620), chr(0x00A0)]
    + [chr(0x0640), chr(0x06DE), chr(0x06E9), chr(0x200F)]
)

def normalize_qalon_text(text):
    text = unicodedata.normalize('NFC', text)
    text = ''.join(ch for ch in text if ch not in _REMOVE_CHARS)
    return ' '.join(text.split())

# Load canonical text
with open(TEXT_FILE, encoding='utf-8') as f:
    quran = json.load(f)
print(f'Loaded {len(quran)} ayahs from qalon_canonical.json')

# Detect already-completed surahs
completed = set()
if os.path.isdir(CHECKPOINT_DIR):
    for fname in os.listdir(CHECKPOINT_DIR):
        if fname.endswith('.json'):
            try:
                completed.add(int(fname.replace('.json', '')))
            except ValueError:
                pass

print(f'Already completed : {len(completed)} / 114 surahs')
print(f'Remaining         : {114 - len(completed)} surahs')
if completed:
    done_list = sorted(completed)
    print(f'Completed surahs  : {done_list}')

# Load pairing manifest
with open(MANIFEST_FILE, encoding='utf-8') as f:
    pairing_manifest = json.load(f)
print(f'Manifest entries  : {len(pairing_manifest)} surahs')

## Cell 7 — What a Segment Is

A segment is one ayah cut from the surah audio. For example, surah 1
(Al-Fatihah) has 7 ayahs. After alignment, we get 7 short WAV files:

```
001_001.wav    Ayah 1:  بِسْمِ اللَّهِ الرَّحْمَٰنِ الرَّحِيمِ
001_002.wav    Ayah 2:  الْحَمْدُ لِلهِ رَبِّ الْعَالَمِينَ
001_003.wav    Ayah 3:  الرَّحْمَٰنِ الرَّحِيمِ
001_004.wav    Ayah 4:  مَالِكِ يَوْمِ الدِّينِ
001_005.wav    Ayah 5:  إِيَّاكَ نَعْبُدُ وَإِيَّاكَ نَسْتَعِينُ
001_006.wav    Ayah 6:  اهْدِنَا الصِّرَاطَ الْمُسْتَقِيمَ
001_007.wav    Ayah 7:  صِرَاطَ الَّذِينَ أَنْعَمْتَ ...
```

Each file contains exactly one ayah of audio, paired with its diacritised
Qaloon text label. This surah-to-ayah cutting is what makes training possible:
the model receives a short audio clip and learns to predict the diacritised
Arabic characters from left to right using CTC.

We add **100 ms of padding** on each side of every boundary. This prevents
cutting off the first or last phoneme of an ayah — a common problem with
hard boundary detection.

## Cell 8 — Load WhisperX Models

We load two models:

- **`large-v3`** — the Whisper ASR model that generates word timestamps.
  We use `float16` (half precision) to fit within the T4's 16 GB VRAM.
- **Arabic alignment model** — a Wav2Vec2 model fine-tuned specifically for
  Arabic phoneme-to-audio alignment. This refines the Whisper timestamps to
  millisecond precision.

Loading takes ~2 minutes on first run (model download). Subsequent runs load
from the Colab disk cache in ~30 seconds.

In [ ]:
import whisperx

device = 'cuda'

model = whisperx.load_model(
    'large-v3',
    device=device,
    compute_type='float16',
    language='ar',
)
align_model, metadata = whisperx.load_align_model(
    language_code='ar',
    device=device,
)
print('Models loaded successfully.')

## Cell 9 — What Happens for Each Surah

The processing loop runs the same six steps for every surah:

**Step 1 — Load the WAV file.**  
The 16 kHz mono WAV from Notebook 01 is read into a float32 numpy array.

**Step 2 — Run WhisperX transcription.**  
Whisper `large-v3` produces a rough transcript with word-level timestamps.
We only keep the timing — the Whisper text is discarded.

**Step 3 — Run WhisperX alignment.**  
The Wav2Vec2 alignment model refines each word's start and end time to
millisecond precision.

**Step 4 — Match timestamps to ayah boundaries.**  
We count words in each Qaloon ayah (by splitting on spaces) and assign
that many consecutive WhisperX words to each ayah in order. If the total
word count from WhisperX differs from the Qaloon count by more than 35%
(due to speech disfluencies or WhisperX errors), we fall back to equal-time
division of the surah across its ayahs.

**Step 5 — Cut and save with padding.**  
Each ayah boundary gets 100 ms of padding on both sides, clamped to
[0, total_duration]. The segment is saved as `SSSAAA.wav` in `SEGMENTS_DIR`.

**Step 6 — Save checkpoint.**  
The segment metadata list is written to `CHECKPOINT_DIR/SSS.json`
immediately. If Colab disconnects after step 6, this surah is marked done
and will be skipped on the next run.

## Cell 10 — The Alignment Function

Two helper functions and the main `align_surah` function. The helpers are
defined here so they can be tested independently before running the full loop.

In [ ]:
import soundfile as sf
import numpy as np

PADDING_SEC = 0.10   # 100 ms added before and after every ayah boundary
SR          = 16_000  # sample rate — must match Notebook 01 output


def _words_to_ayah_boundaries(words, ayah_list, total_duration):
    '''
    Map WhisperX word-level timestamps to ayah-level (start, end) pairs.

    Primary method — word-count matching:
      Count space-separated words in each normalised Qaloon ayah.
      Assign that many consecutive WhisperX words to each ayah in order.

    Fallback method — equal-time division:
      Used when the WhisperX word count differs from the Qaloon word count
      by more than 35%, or when WhisperX produced no usable words.

    Returns (boundaries, method):
      boundaries : list of (start_sec, end_sec) tuples, one per ayah
      method     : "whisperx" or "equal_time"
    '''
    n_ayahs = len(ayah_list)

    # Keep only words that have valid timing from WhisperX
    valid_words = [w for w in words if 'start' in w and 'end' in w]

    # Word counts from the normalised Qaloon text
    ayah_word_counts = [
        max(len(normalize_qalon_text(a['text']).split()), 1)
        for a in ayah_list
    ]
    total_qaloon = sum(ayah_word_counts)
    total_wx     = len(valid_words)

    use_fallback = (
        total_wx == 0
        or total_qaloon == 0
        or (abs(total_wx - total_qaloon) / total_qaloon) > 0.35
    )

    if use_fallback:
        step   = total_duration / n_ayahs
        bounds = [
            (i * step, min((i + 1) * step, total_duration))
            for i in range(n_ayahs)
        ]
        return bounds, 'equal_time'

    # Sequential word assignment
    boundaries = []
    w_idx = 0
    for count in ayah_word_counts:
        chunk = valid_words[w_idx: w_idx + count]
        if chunk:
            s = chunk[0]['start']
            e = chunk[-1]['end']
        elif boundaries:
            # Ran out of WhisperX words — extend proportionally from last end
            remaining_ayahs = n_ayahs - len(boundaries)
            last_end  = boundaries[-1][1]
            step_size = (total_duration - last_end) / max(remaining_ayahs, 1)
            s = last_end + len(boundaries) * step_size - step_size
            e = s + step_size
        else:
            step_size = total_duration / n_ayahs
            s = 0.0
            e = step_size
        boundaries.append((s, e))
        w_idx += count

    return boundaries, 'whisperx'


def align_surah(surah_num, wav_path, ayah_list,
                model, align_model, metadata, device,
                segments_dir, padding_sec=PADDING_SEC):
    '''
    Align one surah WAV into individual ayah-level segment clips.

    Parameters
    ----------
    surah_num    : int   (1-114)
    wav_path     : str   path to the 16 kHz mono WAV file
    ayah_list    : list  [{"key": "001001", "ayah_num": 1, "text": "..."}]
    model        : WhisperX ASR model
    align_model  : WhisperX Wav2Vec2 alignment model
    metadata     : alignment metadata dict
    device       : str   "cuda" or "cpu"
    segments_dir : str   directory to write segment WAV files
    padding_sec  : float seconds of padding on each side of each boundary

    Returns
    -------
    list of dicts, one per ayah, matching segments_manifest.json schema
    '''
    # Step 1 — load audio (float32 numpy array, 16 kHz)
    audio          = whisperx.load_audio(str(wav_path))
    total_duration = len(audio) / SR

    # Step 2 — transcribe (we want timing, not the transcript text)
    result = model.transcribe(audio, batch_size=16)

    # Step 3 — refine to word-level timestamps
    aligned = whisperx.align(
        result['segments'], align_model, metadata, audio, device,
        return_char_alignments=False,
    )
    words = aligned.get('word_segments', [])

    # Step 4 — map words to ayah boundaries
    boundaries, method = _words_to_ayah_boundaries(
        words, ayah_list, total_duration
    )

    # Step 5 — cut, pad, and save each ayah segment
    segments = []
    for ayah_info, (raw_start, raw_end) in zip(ayah_list, boundaries):
        start = max(0.0, raw_start - padding_sec)
        end   = min(total_duration, raw_end  + padding_sec)

        s_idx = int(start * SR)
        e_idx = int(end   * SR)
        clip  = audio[s_idx: e_idx]

        key      = ayah_info['key']          # e.g. '001001'
        out_name = f'{key[:3]}_{key[3:]}.wav'
        out_path = os.path.join(segments_dir, out_name)
        sf.write(out_path, clip, SR)

        segments.append({
            'ayah_key'         : key,
            'surah'            : surah_num,
            'ayah'             : ayah_info['ayah_num'],
            'text'             : normalize_qalon_text(ayah_info['text']),
            'wav_path'         : out_path,
            'duration_sec'     : round(end - start, 3),
            'alignment_method' : method,
        })

    return segments


print('align_surah and helpers defined.')

## Cell 11 — Main Processing Loop

Iterates over all 114 surahs. Skips completed ones, processes the rest,
and saves a checkpoint immediately after each success. Failures are caught
per-surah — one bad surah does not stop the rest.

In [ ]:
results = []
failed  = []

for surah_num in range(1, 115):
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f'{surah_num:03d}.json')

    # Resume logic: skip surahs that already have a checkpoint
    if os.path.exists(checkpoint_path):
        print(f'Surah {surah_num:03d}: already done, skipping')
        continue

    wav_path = os.path.join(WAV_DIR, f'{surah_num:03d}.wav')
    if not os.path.exists(wav_path):
        print(f'Surah {surah_num:03d}: WAV not found, skipping')
        failed.append(surah_num)
        continue

    # Build ordered ayah list for this surah
    prefix    = f'{surah_num:03d}'
    ayah_list = sorted(
        [{'key': k, 'ayah_num': int(k[3:]), 'text': v}
         for k, v in quran.items() if k.startswith(prefix)],
        key=lambda x: x['ayah_num'],
    )
    if not ayah_list:
        print(f'Surah {surah_num:03d}: no text entries, skipping')
        failed.append(surah_num)
        continue

    try:
        segments = align_surah(
            surah_num, wav_path, ayah_list,
            model, align_model, metadata,
            device='cuda',
            segments_dir=SEGMENTS_DIR,
        )

        # Step 6 — save checkpoint immediately
        with open(checkpoint_path, 'w', encoding='utf-8') as f:
            json.dump(segments, f, ensure_ascii=False, indent=2)

        results.extend(segments)
        meth = segments[0]['alignment_method'] if segments else 'n/a'
        print(f'Surah {surah_num:03d}: {len(segments)} segments saved  [{meth}]')

    except Exception as e:
        print(f'Surah {surah_num:03d}: FAILED — {e}')
        failed.append(surah_num)
        continue

print()
print(f'Loop complete.  {len(results)} new segments this session.')
print(f'Failed surahs : {failed}')

## Cell 12 — Build the Master Segments Manifest

After the loop finishes (or after a resume that completes all remaining
surahs), we collect every checkpoint file and merge them into one master
manifest. This manifest is the input to Notebook 04 (dataset construction).

In [ ]:
all_segments = []
for fname in sorted(os.listdir(CHECKPOINT_DIR)):
    if not fname.endswith('.json'):
        continue
    fpath = os.path.join(CHECKPOINT_DIR, fname)
    with open(fpath, encoding='utf-8') as f:
        all_segments.extend(json.load(f))

total_segments = len(all_segments)

if total_segments == 0:
    print('No segments found. Run Cell 11 first.')
else:
    total_sec   = sum(s['duration_sec'] for s in all_segments)
    avg_sec     = total_sec / total_segments
    total_hours = total_sec / 3600

    manifest_path = os.path.join(DRIVE_BASE, 'processed', 'segments_manifest.json')
    with open(manifest_path, 'w', encoding='utf-8') as f:
        json.dump(all_segments, f, ensure_ascii=False, indent=2)

    print(f'Total segments   : {total_segments}')
    print(f'Total duration   : {total_hours:.2f} hours')
    print(f'Average duration : {avg_sec:.2f} s per segment')
    print(f'Manifest saved   : {manifest_path}')

## Cell 13 — What to Do If Some Surahs Failed

If the `failed` list is not empty after Cell 11, do not worry. The most
common causes are:

- **Temporary GPU memory error** — WhisperX ran out of VRAM on a long surah.
  Usually self-resolves on retry because GPU memory is freed between surahs.
- **Network timeout** — Drive write briefly failed.
- **WAV not found** — the surah MP3 was not converted in Notebook 01.

**To retry failed surahs:** simply re-run this notebook from Cell 11.
Because checkpoints are written immediately after each success, the notebook
will skip all completed surahs and attempt only the ones that previously
failed. No segments are duplicated or overwritten.

If a surah fails consistently across three or more re-runs, note the surah
number and the error message. A surah that persistently fails should be
excluded from the dataset rather than holding up the rest of training.

## Cell 14 — Quality Check on Segments

We verify that segment counts match expected ayah counts, inspect the
duration distribution, and flag any segments that are suspiciously short
(under 0.5 s usually indicates a failed alignment at an ayah boundary).

In [ ]:
import random
from collections import defaultdict

if 'all_segments' not in vars() or not all_segments:
    print('No segments available. Run Cells 11 and 12 first.')
else:
    # ── segment count vs expected ayah count ─────────────────────────────────
    expected_per_surah = defaultdict(int)
    for k in quran:
        expected_per_surah[int(k[:3])] += 1

    got_per_surah = defaultdict(int)
    for seg in all_segments:
        got_per_surah[seg['surah']] += 1

    mismatches = [
        (s, expected_per_surah[s], got_per_surah[s])
        for s in range(1, 115)
        if got_per_surah[s] > 0 and got_per_surah[s] != expected_per_surah[s]
    ]
    if mismatches:
        print(f'Segment count mismatches in {len(mismatches)} surahs:')
        for s, exp, got in mismatches[:20]:
            print(f'  Surah {s:03d}: expected {exp}, got {got}')
    else:
        print('Segment counts: all processed surahs match expected ayah counts  OK')

    # ── duration histogram ───────────────────────────────────────────────────
    bin_counts = {'under_2s': 0, 'two_to_10s': 0, 'ten_to_20s': 0, 'over_20s': 0}
    too_short  = []
    for seg in all_segments:
        d = seg['duration_sec']
        if d < 0.5:
            too_short.append(seg['ayah_key'])
        if d < 2:
            bin_counts['under_2s'] += 1
        elif d < 10:
            bin_counts['two_to_10s'] += 1
        elif d < 20:
            bin_counts['ten_to_20s'] += 1
        else:
            bin_counts['over_20s'] += 1

    total = len(all_segments)
    labels = {
        'under_2s'  : '< 2 s',
        'two_to_10s': '2-10 s',
        'ten_to_20s': '10-20 s',
        'over_20s'  : '> 20 s',
    }
    print()
    print('Duration histogram:')
    for key, label in labels.items():
        count = bin_counts[key]
        pct   = count / total * 100
        print(f'  {label:<9}: {count:>5} segments  ({pct:.1f}%)')

    if too_short:
        print(f'Segments under 0.5 s : {len(too_short)}  (first 10: {too_short[:10]})')
    else:
        print('Segments under 0.5 s : 0  OK')

    # ── 5 random samples ─────────────────────────────────────────────────────
    print()
    print('5 random segments:')
    for seg in random.sample(all_segments, min(5, total)):
        key   = seg['ayah_key']
        dur   = seg['duration_sec']
        meth  = seg['alignment_method']
        text  = seg['text'][:55]
        print(f'  [{key}]  {dur:.2f}s  [{meth}]  {text}')

## Cell 15 — Summary and What Notebook 04 Will Do

### What this notebook produced

| Output | Path | Description |
|--------|------|-------------|
| Ayah-level WAV segments | `processed/segments/SSS_AAA.wav` | One file per ayah, 16 kHz, ~2–15 s each |
| Per-surah checkpoints | `processed/alignment_checkpoints/SSS.json` | Enables safe resumption after disconnection |
| Master segments manifest | `processed/segments_manifest.json` | Full list of all segments with text and paths |
| Completion marker | `processed/notebook_03_complete.json` | Summary with counts and duration |

### What Notebook 04 (Training) will do

Notebook 04 reads `segments_manifest.json` and `vocab.json` to construct a
HuggingFace `DatasetDict`. It then fine-tunes
`facebook/wav2vec2-large-xlsr-53-arabic` on the training split using CTC loss.

Key steps in Notebook 04:

1. Load each segment WAV and extract `input_values` via the Wav2Vec2 processor
2. Tokenise each text label using `vocab.json` into integer sequences
3. Fine-tune with `Wav2Vec2ForCTC` and the `Trainer` API
4. Evaluate on the validation split every 400 steps
5. Save the best checkpoint by Word Error Rate

The protected test set (22 ayahs from Notebook 02) is not touched until
Notebook 05 (evaluation).

In [ ]:
import json, os

if 'all_segments' not in vars() or not all_segments:
    print('No segments available. Run Cells 11 and 12 first.')
else:
    total_sec  = sum(s['duration_sec'] for s in all_segments)
    avg_sec    = total_sec / len(all_segments)
    failed_list = failed if 'failed' in vars() else []

    summary = {
        'notebook'             : '03_alignment',
        'status'               : 'complete',
        'total_segments'       : len(all_segments),
        'total_duration_hours' : round(total_sec / 3600, 3),
        'failed_surahs'        : failed_list,
        'average_duration_sec' : round(avg_sec, 3),
    }

    marker_path = os.path.join(DRIVE_BASE, 'processed', 'notebook_03_complete.json')
    with open(marker_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2)

    print('Notebook 03 complete.')
    print(json.dumps(summary, indent=2))
    print(f'Marker saved to: {marker_path}')